# test Music voice track recognition accuracy

## download video
SAINt JHN Lust

In [2]:
from hipHopProducer import HipHopAutoProject

app = HipHopAutoProject()
app.download_step("SAINt JHN Lust", output_path="~/Downloads/SAINt_JHN_Lust.mp4")


🏗️ 正在初始化模型...


llama_model_load_from_file_impl: using device Metal (Apple M4) - 7241 MiB free
llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from /Users/randy/.cache/huggingface/hub/models--Randyliu99--qwen2.5-7b-jcole-gguf/snapshots/797a41ca8b8b3b1335a08e43e3b4f1bb52bb1ae3/./Qwen2.5-7B-Instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Unsloth_Gguf_M7Jqshyv
llama_model_loader: - kv   3:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   4:                         general.size_label str              = 7.6B
llama_model_loader: - kv   5:               

📥 正在搜索并下载: SAINt JHN Lust...
[generic] Extracting URL: SAINt JHN Lust
[youtube:search] Extracting URL: ytsearch1:SAINt JHN Lust
[download] Downloading playlist: SAINt JHN Lust
[youtube:search] query "SAINt JHN Lust": Downloading web client config
[youtube:search] query "SAINt JHN Lust" page 1: Downloading API JSON
[youtube:search] Playlist SAINt JHN Lust: Downloading 1 items of 1
[download] Downloading item 1 of 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=N8Ykxh9n5Hg
[youtube] N8Ykxh9n5Hg: Downloading webpage


[youtube] N8Ykxh9n5Hg: Downloading android vr player API JSON
[info] N8Ykxh9n5Hg: Downloading 1 format(s): 399+140
[download] Destination: /Users/randy/Downloads/SAINt_JHN_Lust.f399.mp4
[download] 100% of   18.21MiB in 00:00:02 at 6.09MiB/s   
[download] Destination: /Users/randy/Downloads/SAINt_JHN_Lust.f140.m4a
[download] 100% of    2.69MiB in 00:00:00 at 5.30MiB/s   
[Merger] Merging formats into "/Users/randy/Downloads/SAINt_JHN_Lust.mp4"
Deleting original file /Users/randy/Downloads/SAINt_JHN_Lust.f399.mp4 (pass -k to keep)
Deleting original file /Users/randy/Downloads/SAINt_JHN_Lust.f140.m4a (pass -k to keep)
[download] Finished downloading playlist: SAINt JHN Lust


'~/Downloads/SAINt_JHN_Lust.mp4'

## extract the voice track

In [ ]:
from faster_whisper import WhisperModel
import subprocess
import os

import torch

whisper = WhisperModel("medium", device="cpu", compute_type="int8")


def transcribe_step1(video_path, audio_path):
    temp_dir = '/User/randy/Downloads/temp_audio'
    raw_audio = os.path.join(temp_dir, "raw_audio.wav")
    print("正在提取音轨...")
    subprocess.run([
        'ffmpeg', '-i', video_path, 
        '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', 
        audio_path, '-y'
    ], check=True)

    temp_dir = '/Users/randy/Downloads/temp_audio'
    demucs_cmd = [
        "python3", "-m", "demucs",
        "-o", temp_dir,
        "--two-stems", "vocals",
        "-d", "mps",  # 既然你命令行 mps 成功了，这里保持一致
        raw_audio
    ]
    # 增加 check=True 和 capture_output=True 来获取错误细节
    result = subprocess.run(demucs_cmd, check=True, capture_output=True, text=True)
    audio_path = os.path.join(temp_dir, "/htdemucs/temp_audio/vocals.wav")
    print("🎙️ 正在提取音轨并识别 (Faster-Whisper)...")
    # vad_filter=True 能有效过滤掉嘻哈伴奏里的幻听
    segments, _ = whisper.transcribe(
        audio_path,
        # 1. 简化的 Prompt：只给关键词，不给完整句子
        initial_prompt="Rap, Hip-hop, lyrics, slang.", 
        
        # 2. 提高采样门槛：如果 AI 不确定，就不要乱说话
        beam_size=5,
        best_of=5,
        
        # 3. 关键：设置静音过滤和压缩比限制
        vad_filter=True,  # 确保开启静音过滤
        vad_parameters=dict(min_silence_duration_ms=500), # 超过0.5秒没声音就跳过
        
        # 4. 幻听控制：如果这一段重复率太高，丢弃它
        compression_ratio_threshold=2.4, 
        no_speech_threshold=0.6,
        
        word_timestamps=True
    )
    full_data = []
    # 纯英文文本列表（用于交给翻译引擎）
    english_texts_only = []

    for segment in segments:
        clean_text = segment.text.strip()
        if clean_text:
            # 存储详细信息
            full_data.append({
                'start': segment.start,
                'end': segment.end,
                'text': clean_text
            })
            # 存储纯文本
            english_texts_only.append(clean_text)
            # print(f"已提取: {clean_text}")

    # 清理临时音频
    # if os.path.exists(audio_path):
    #     os.remove(audio_path)

    # 返回这两个列表供后续使用
    return full_data, english_texts_only

segs, english_texts = transcribe_step1("/Users/randy/Downloads/SAINt_JHN_Lust.mp4", "temp_audio.wav")
print("提取的英文文本：", english_texts)

正在提取音轨...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.4.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --cc=clang --host-cflags= --host-ldflags= --enable-gpl --enable-libaom --enable-libdav1d --enable-libharfbuzz --enable-libmp3lame --enable-libopus --enable-libsnappy --enable-libsvtav1 --enable-libtheora --enable-libvorbis --enable-libvpx --enable-libx264 --enable-libx265 --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-demuxer=dash --enable-neon --enable-opencl --enable-audiotoolbox --enable-videotoolbox --disable-htmlpages
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/User

✨ 正在使用 Demucs 分离人声 (排除背景重低音干扰)...
❌ Demucs 运行失败: Command '['python3', '-m', 'demucs.separate', '-d', 'mps', '--two-stems', 'vocals', '-o', '/User/randy/Downloads/temp_audio', '/User/randy/Downloads/temp_audio/raw_audio.wav']' returned non-zero exit status 1.，将直接识别原始音频
🎙️ 正在提取音轨并识别 (Faster-Whisper)...


KeyboardInterrupt: 

# use demucs to separate vocal and non-vocal voice

In [20]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
temp_dir = '/Users/randy/Downloads/temp_audio'
raw_audio = "/Users/randy/Downloads/temp_audio.wav" 
demucs_cmd = [
    "python3", "-m", "demucs",
    "-o", temp_dir,
    "--two-stems", "vocals",
    "-d", "mps",  # 既然你命令行 mps 成功了，这里保持一致
    raw_audio
]
try:
    # 增加 check=True 和 capture_output=True 来获取错误细节
    result = subprocess.run(demucs_cmd, check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as e:
    print("❌ Demucs 运行失败！")
    print(f"标准输出: {e.stdout}")
    print(f"错误详情 (stderr): {e.stderr}") # 重点看这里
    raise e

In [22]:
segments, _ = whisper.transcribe(
    "/Users/randy/Downloads/temp_audio/htdemucs/temp_audio/vocals.wav",
    # 1. 简化的 Prompt：只给关键词，不给完整句子
    initial_prompt="Rap, Hip-hop, lyrics, slang.", 
    
    # 2. 提高采样门槛：如果 AI 不确定，就不要乱说话
    beam_size=5,
    best_of=5,
    
    # 3. 关键：设置静音过滤和压缩比限制
    vad_filter=True,  # 确保开启静音过滤
    vad_parameters=dict(min_silence_duration_ms=500), # 超过0.5秒没声音就跳过
    
    # 4. 幻听控制：如果这一段重复率太高，丢弃它
    compression_ratio_threshold=2.4, 
    no_speech_threshold=0.6,
    
    word_timestamps=True
)
full_data = []
# 纯英文文本列表（用于交给翻译引擎）
english_texts_only = []

for segment in segments:
    clean_text = segment.text.strip()
    if clean_text:
        # 存储详细信息
        full_data.append({
            'start': segment.start,
            'end': segment.end,
            'text': clean_text
        })
        # 存储纯文本
        english_texts_only.append(clean_text)

print("提取的英文文本：", english_texts_only)

提取的英文文本： ["Young puss playin' war games, she wanted love, but it only made more pain.", "Young puss playin' war games, she wanted love, but it only made more pain.", "Picture my soul, climbin' out an infinite hole, where niggas die over pride and live for the dope.", "If I survive, then I strive to hit with the flow, hopin' these waves are paid, but the ripples are slow.", "And I'm conflicted with no digits to show, but fantasies of frivolous hoes drippin' this pole, the tempest is slow.", "But you know that ain't stoppin' lil' nigga from slidin' like the tides on a whip in the snow.", "This is for show, I'm tip-toin' around the abyss with sticks blowin'.", "Pitch pole with a tint that's a glyph, the wrist glowin', face tits from rent owein'.", "Fist clenched when niggas diss, but since knowin', lives expire when niggas pride gets hide.", 'In the vocal range, the cyberized vocal chain, uh, my road to fame is right with fights and broken lanes.', "And tolls I can't afford, but I won't c